# Devflow — Voice-Powered Developer Intent Classifier
**Pipeline:** Voice → Whisper ASR → `[THIS NOTEBOOK]` → Intent-Classified JSON → LLM

### Notebook Sections
1. Install & Imports
2. Dataset Generation (Gemini 2.5 Pro API)
3. Preprocessing Pipeline (6 steps)
4. Model A — LinearSVC Baseline
5. Model B — DistilBERT Fine-tuning
6. Entity Extractor (spaCy EntityRuler)
7. Full Inference Pipeline (JSON output)
8. Evaluation — Macro F1, Ablation, Cohen's Kappa
9. Save Artefacts for Web App Integration

## 1. Install & Imports

In [1]:
# ── Install all dependencies ──────────────────────────────────────────────────
%pip install -q google-genai groq nltk scikit-learn transformers datasets \
               spacy sentence-transformers pandas scipy wandb \
               accelerate torch nltk
!python -m spacy download en_core_web_sm -q

import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
# nltk.download('punkt', quiet=True)
# nltk.download('punkt_tab', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('averaged_perceptron_tagger', quiet=True)
nltk.download('averaged_perceptron_tagger_eng', quiet=True)
print('✅ All dependencies installed')

Note: you may need to restart the kernel to use updated packages.
✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
✅ All dependencies installed


[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\sampu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\sampu\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [2]:
# ── Core imports ──────────────────────────────────────────────────────────────
import os, re, json, random, time, warnings
import pandas as pd
import numpy as np
from pathlib import Path
from collections import defaultdict

# NLP
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from nltk import pos_tag
from nltk.corpus import wordnet

# ML
from sklearn.svm import LinearSVC
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.pipeline import Pipeline
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, f1_score,
    precision_recall_fscore_support, cohen_kappa_score
)
from sklearn.calibration import CalibratedClassifierCV
from scipy import stats

# Transformers
import torch
from transformers import (
    AutoTokenizer, AutoModelForSequenceClassification,
    TrainingArguments, Trainer
)
from datasets import Dataset

# spaCy
import spacy
from spacy.pipeline import EntityRuler

# Gemini (new google-genai SDK)
from google import genai
from google.genai import types as genai_types

warnings.filterwarnings('ignore')
SEED = 42
random.seed(SEED)
np.random.seed(SEED)

INTENTS = ['debug', 'generate', 'refactor', 'explain', 'scaffold', 'test', 'document']
DEVICE  = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'✅ Imports done | Device: {DEVICE}')

✅ Imports done | Device: cpu


## 2. Dataset Generation via gemini-2.5-flash-lite API

In [ ]:
# ── API Setup ─────────────────────────────────────────────────────────────────
from groq import Groq

GROQ_API_KEY = ''
GROQ_MODEL   = 'llama-3.3-70b-versatile'   # free, 14400 req/day, 6000 RPM

groq_client = Groq(api_key=GROQ_API_KEY)
print(f'✅ Groq client ready → {GROQ_MODEL}')

✅ Groq client ready → llama-3.3-70b-versatile


In [ ]:
# # ── Set your Gemini API key ──────────────────────────────────────────────────
# import os

# # Option 1: Set as environment variable (recommended)  
# # In your terminal: set GEMINI_API_KEY=AIza...
# # Option 2: Paste directly below (less secure)

# GEMINI_API_KEY = ''
# # os.environ.get('GEMINI_API_KEY', 'AIza-YOUR_KEY_HERE')

# if GEMINI_API_KEY == 'AIza-YOUR_KEY_HERE':
#     print('⚠️  Using placeholder key — set GEMINI_API_KEY env variable or paste your key above')
# else:
#     print('✅ Loaded GEMINI_API_KEY')

# # google-genai SDK client
# gemini_client = genai.Client(api_key=GEMINI_API_KEY)

# # gemini-2.0-flash: free tier — 15 RPM, 1500 RPD, 1M TPM
# GEMINI_MODEL = 'gemini-2.5-flash-lite'
# print(f'✅ Gemini client ready → {GEMINI_MODEL} (google-genai SDK)')

In [5]:
# ── Seed utterances: 7–8 per class, hand-crafted for quality ─────────────────
# Gemini will paraphrase each seed 4–5x with realistic voice noise

SEED_UTTERANCES = {
    'debug': [
        'fix the null pointer exception in login service',
        'why is my authentication module throwing a 403 error',
        'debug the memory leak in the payment processor',
        'the api is returning 500 on every post request',
        'find the bug causing infinite loop in data pipeline',
        'trace the runtime exception in the order handler',
        'resolve the type error in user profile component',
        'the database connection keeps timing out fix it',
        'the mobile app crashes on android when uploading an image',
        'my docker container exits with code 137 every few hours',
        'the kafka consumer is lagging behind by a million messages',
        'pandas is throwing a key error when i merge these two dataframes',
    ],
    'generate': [
        'write a python function to parse json from an api response',
        'create a rest endpoint for user registration in express',
        'generate a sql query to find top ten customers by revenue',
        'write a react component for a dropdown menu',
        'create a bash script to automate daily backups',
        'generate a typescript interface for the user model',
        'write a middleware for jwt authentication in django',
        'create a websocket handler for real time notifications',
        'write a terraform script to spin up an ec2 instance',
        'generate a github actions workflow for ci cd pipeline',
        'write a pandas script to clean and normalize this csv file',
        'create a flutter widget for a bottom navigation bar',
    ],
    'refactor': [
        'refactor the authentication module to use dependency injection',
        'clean up the database service it has too many nested callbacks',
        'convert the user controller from class based to functional',
        'simplify the payment logic it is too complex',
        'extract the validation logic into a separate utility file',
        'rename all variables in the config module to be more descriptive',
        'break down the monolithic api into smaller services',
        'remove duplicated code in the data transformation layer',
        'convert these promise chains to async await in the auth service',
        'move all hardcoded strings to a constants file',
        'split this thousand line file into separate modules',
        'replace the switch statement with a strategy pattern',
    ],
    'explain': [
        'explain how the oauth flow works in this codebase',
        'what does the middleware stack do in this express app',
        'how does the caching layer interact with the database',
        'explain the difference between promises and async await',
        'why are we using redux instead of context api here',
        'what is the purpose of the abstract factory pattern here',
        'how does the event emitter work in this module',
        'explain the data flow from the frontend to the backend',
        'what is the difference between docker and kubernetes',
        'why is the build so slow what is webpack actually doing',
        'explain what this regex pattern is matching',
        'how does database indexing work in postgres',
    ],
    'scaffold': [
        'set up a new react project with typescript and tailwind',
        'initialise a django rest framework project with auth',
        'create a new microservice boilerplate for the payments team',
        'scaffold a nodejs express app with mongoose and dotenv',
        'set up the folder structure for a clean architecture project',
        'bootstrap a new flutter app with firebase integration',
        'create a monorepo setup with shared component library',
        'initialise a fastapi project with postgresql and alembic',
        'set up a new data science project with jupyter and conda',
        'create a terraform project structure for aws infrastructure',
        'bootstrap a new android app with jetpack compose',
        'set up a grafana and prometheus monitoring stack',
    ],
    'test': [
        'write unit tests for the login service using jest',
        'create integration tests for the payment api endpoints',
        'add test coverage for the user authentication module',
        'write end to end tests for the checkout flow using cypress',
        'generate mock data for testing the order processing pipeline',
        'write a pytest fixture for the database connection',
        'create snapshot tests for all dashboard components',
        'add property based tests for the data validation functions',
        'write contract tests between the frontend and backend api',
        'add load tests for the search endpoint using locust',
        'mock the stripe api for payment service unit tests',
        'write tests for all the edge cases in the cart calculation',
    ],
    'document': [
        'write jsdoc comments for all public functions in the api module',
        'generate a readme for the authentication service',
        'document the rest api endpoints with openapi spec',
        'add inline comments to the payment processing logic',
        'write a contributing guide for the open source repo',
        'create architecture documentation for the microservices setup',
        'document the environment variables needed to run the project',
        'write changelog entries for the latest release',
        'write a runbook for when the payment service goes down',
        'document the database schema and all the table relationships',
        'add docstrings to all the data processing functions in python',
        'create a postman collection for all the api endpoints',
    ],
}

total_seeds = sum(len(v) for v in SEED_UTTERANCES.values())
print(f'✅ Seed utterances ready: {total_seeds} seeds across {len(INTENTS)} classes')

✅ Seed utterances ready: 84 seeds across 7 classes


In [6]:
# ── Generation helpers ────────────────────────────────────────────────────────

# ENTITY_CONTEXT = {
#     'debug':    ['java', 'python', 'typescript', 'go', 'spring', 'django', 'express', 'NullPointerException', 'TypeError', '404', '500', 'runtime exception'],
#     'generate': ['python', 'typescript', 'java', 'rust', 'go', 'react', 'express', 'django', 'fastapi', 'spring'],
#     'refactor': ['python', 'typescript', 'java', 'kotlin', 'react', 'angular', 'spring', 'django'],
#     'explain':  ['python', 'javascript', 'java', 'react', 'redux', 'express', 'django', 'spring'],
#     'scaffold': ['react', 'nextjs', 'django', 'fastapi', 'express', 'flutter', 'spring', 'typescript'],
#     'test':     ['jest', 'pytest', 'cypress', 'junit', 'mocha', 'react testing library'],
#     'document': ['jsdoc', 'sphinx', 'openapi', 'markdown', 'typescript', 'python'],
# }

# UPDATED:
ENTITY_CONTEXT = {
    'debug': [
        'java', 'python', 'typescript', 'go', 'kotlin', 'android',
        'spring', 'django', 'express', 'fastapi',
        'NullPointerException', 'TypeError', 'KeyError',
        '404', '500', '137',
        'runtime exception', 'memory leak', 'infinite loop', 'timeout',
        'docker', 'kafka', 'pandas', 'postgres',
    ],
    'generate': [
        'python', 'typescript', 'java', 'rust', 'go', 'dart', 'bash',
        'react', 'flutter', 'express', 'django', 'fastapi', 'spring',
        'terraform', 'aws', 'github actions', 'pandas', 'sql',
    ],
    'refactor': [
        'python', 'typescript', 'java', 'kotlin',
        'react', 'angular', 'spring', 'django',
        'async await', 'dependency injection', 'strategy pattern',
        'constants', 'modules', 'promise',
    ],
    'explain': [
        'python', 'javascript', 'java', 'go',
        'react', 'redux', 'express', 'django', 'spring',
        'docker', 'kubernetes', 'webpack', 'postgres',
        'oauth', 'jwt', 'regex', 'indexing', 'caching',
    ],
    'scaffold': [
        'react', 'nextjs', 'django', 'fastapi', 'express',
        'flutter', 'spring', 'typescript', 'android',
        'terraform', 'aws', 'jupyter', 'conda',
        'grafana', 'prometheus', 'jetpack compose', 'firebase',
    ],
    'test': [
        'jest', 'pytest', 'cypress', 'junit', 'mocha',
        'react testing library', 'vitest', 'locust',
        'stripe', 'contract testing', 'property based testing',
        'snapshot', 'integration', 'load testing', 'mock',
    ],
    'document': [
        'jsdoc', 'sphinx', 'openapi', 'markdown',
        'typescript', 'python', 'postman',
        'readme', 'runbook', 'changelog', 'docstring',
        'swagger', 'database schema', 'architecture',
    ],
}


# ── Rate limit config ─────────────────────────────────────────────────────────
# gemini-2.0-flash free tier: 15 RPM → 1 req per 4s is safe
# We use 5s gap to have headroom and avoid burst issues
MIN_REQUEST_GAP_SECS = 5.0
_last_request_ts = 0.0

def _throttle():
    """Sleep until MIN_REQUEST_GAP_SECS has elapsed since the last call."""
    global _last_request_ts
    gap = time.time() - _last_request_ts
    if gap < MIN_REQUEST_GAP_SECS:
        time.sleep(MIN_REQUEST_GAP_SECS - gap)
    _last_request_ts = time.time()

def _parse_retry_after(err: str) -> float:
    """Pull the 'retry in Xs' wait time from a 429 message, else return 65s."""
    m = re.search(r'retry\s+in\s+([\d.]+)s', err, re.IGNORECASE)
    return float(m.group(1)) + 3.0 if m else 65.0

def build_prompt(intent: str, seed: str, n: int = 14) -> str:
    entities = ', '.join(random.sample(ENTITY_CONTEXT[intent], min(4, len(ENTITY_CONTEXT[intent]))))
    short_n  = n // 4
    medium_n = n // 3
    long_n   = n - short_n - medium_n - 2
    noisy_n  = 2   # extra noisy/realistic variants

# short_n  = 14 // 4 = 3     (integer division, floor)
# medium_n = 14 // 3 = 4     (integer division, floor)
# noisy_n  = 2               (hardcoded always)
# long_n   = 14 - 3 - 4 - 2 = 5

# Total check: 3 + 4 + 5 + 2 = 14 ✅

    return f"""You are generating synthetic training data for a developer voice NLP classifier.

Intent class : {intent.upper()}
Seed         : \"{seed}\"
Tech hints   : {entities}  (use 0-2 per variant, not mandatory)

Generate exactly {n} paraphrased variants distributed as:
  {short_n} SHORT   (3-8 words)    e.g. "fix null pointer login"
  {medium_n} MEDIUM  (9-17 words)   e.g. "uh can you fix the null pointer in the login service"
  {long_n} LONG    (18-25 words)  e.g. "um so every time i try to log in the service throws a null pointer and i have no clue why"
  {noisy_n} NOISY   (any length, heavy fillers, fragmented, like someone thinking aloud)

All variants must:
  - Unambiguously belong to {intent.upper()} intent
  - Sound like a developer speaking aloud — casual, spontaneous, NOT formal
  - Be lowercase only, no punctuation
  - Each must differ structurally from the others
  - No numbering, no labels, no explanation

Return ONLY a flat JSON array of {n} strings. No markdown fences. No preamble."""


def generate_variants(intent: str, seed: str, n: int = 14, max_retries: int = 5) -> list:
    """Call Groq (LLaMA 3.3 70B) with 429-aware retry."""
    prompt = build_prompt(intent, seed, n)

    for attempt in range(max_retries):
        _throttle()
        try:
            response = groq_client.chat.completions.create(
                model=GROQ_MODEL,
                messages=[{"role": "user", "content": prompt}],
                temperature=0.9,
                max_tokens=1024,
            )
            raw = response.choices[0].message.content.strip()
            raw = re.sub(r'```(?:json)?', '', raw).strip().rstrip('`').strip()
            variants = json.loads(raw)
            if isinstance(variants, list) and variants:
                return [v.strip().lower() for v in variants if isinstance(v, str) and v.strip()]
        except Exception as e:
            err = str(e)
            if '429' in err:
                wait = _parse_retry_after(err)
                print(f'  ⏳ 429 rate limit — waiting {wait:.0f}s (attempt {attempt+1}/{max_retries})')
                time.sleep(wait)
            else:
                print(f'  ⚠️  Non-429 error (attempt {attempt+1}): {err[:140]}')
                time.sleep(2)

    print('  ❌ All retries exhausted — skipping seed')
    return []

# Used when Gemini
# def generate_variants(intent: str, seed: str, n: int = 9, max_retries: int = 6) -> list:
#     """Call Gemini via the new google-genai SDK with 429-aware retry."""
#     prompt = build_prompt(intent, seed, n)

#     for attempt in range(max_retries):
#         _throttle()   # always enforce minimum gap before hitting the API
#         try:
#             response = gemini_client.models.generate_content(
#                 model=GEMINI_MODEL,
#                 contents=prompt,
#                 config=genai_types.GenerateContentConfig(
#                     temperature=0.95,
#                     max_output_tokens=1024,
#                 ),
#             )
#             raw = response.text.strip()
#             raw = re.sub(r'```(?:json)?', '', raw).strip().rstrip('`').strip()
#             variants = json.loads(raw)
#             if isinstance(variants, list) and variants:
#                 return [v.strip().lower() for v in variants if isinstance(v, str) and v.strip()]
#         except Exception as e:
#             err = str(e)
#             if '429' in err:
#                 wait = _parse_retry_after(err)
#                 print(f'  ⏳ 429 rate limit — waiting {wait:.0f}s (attempt {attempt+1}/{max_retries})')
#                 time.sleep(wait)
#             else:
#                 print(f'  ⚠️  Non-429 error (attempt {attempt+1}): {err[:140]}')
#                 time.sleep(4)

#     print(' ❌ All retries exhausted — skipping seed')
#     return []


print(f'✅ Generation helpers ready')
print(f'   SDK            : groq (LLaMA 3.3 70B)')
print(f'   Model          : {GROQ_MODEL}')
print(f'   Request gap    : {MIN_REQUEST_GAP_SECS}s → ~{60/MIN_REQUEST_GAP_SECS:.0f} RPM')
print(f'   Limits         : 30 RPM | 1K RPD | 12K TPM | 100K TPD')
print(f'   Total API calls: {sum(len(v) for v in SEED_UTTERANCES.values())} seeds')
print(f'   Est. runtime   : ~{sum(len(v) for v in SEED_UTTERANCES.values()) * MIN_REQUEST_GAP_SECS / 60:.1f} min')
print(f'   Target records : ~{sum(len(v) for v in SEED_UTTERANCES.values()) * 15}')

✅ Generation helpers ready
   SDK            : groq (LLaMA 3.3 70B)
   Model          : llama-3.3-70b-versatile
   Request gap    : 5.0s → ~12 RPM
   Limits         : 30 RPM | 1K RPD | 12K TPM | 100K TPD
   Total API calls: 84 seeds
   Est. runtime   : ~7.0 min
   Target records : ~1260


In [7]:
# ── Quick API key sanity check ────────────────────────────────────────────────
import sys
print("Testing Groq API key...")
sys.stdout.flush()

try:
    test_response = groq_client.chat.completions.create(
        model=GROQ_MODEL,
        messages=[{"role": "user", "content": "Say hello in 3 words."}],
        max_tokens=20,
    )
    print(f"✅ Key works! Response: {test_response.choices[0].message.content.strip()}")
except Exception as e:
    print(f"❌ Failed: {e}")

Testing Groq API key...
✅ Key works! Response: Hello to you.


ONCE DATASET GENERATED 
<br/> No need to run again n again

In [9]:
# ── Run dataset generation ────────────────────────────────────────────────────
# 56 seeds × 9 variants + 56 seeds = ~560 API calls + seeds = ~2,000 examples
# Free tier: 15 req/min → MIN_REQUEST_GAP=4.5s → ~4 min per class, ~28 min total
# If you hit a 429 mid-run, the retry logic will wait the exact time Gemini asks for
# and then resume — no data is lost, the loop continues from where it left off.

VARIANTS_PER_SEED = 14   # 3 short + 3 medium + 3 long per seed
records = []            # accumulates {utterance, intent} dicts

for intent, seeds in SEED_UTTERANCES.items():
    print(f'\n🔄 [{intent.upper()}] — {len(seeds)} seeds × {VARIANTS_PER_SEED} variants')
    intent_count = 0

    for i, seed in enumerate(seeds):
        records.append({'utterance': seed, 'intent': intent})   # keep clean seed

        variants = generate_variants(intent, seed, n=VARIANTS_PER_SEED)
        for v in variants:
            records.append({'utterance': v, 'intent': intent})
        intent_count += len(variants) + 1

        print(f'  [{i+1}/{len(seeds)}] "{seed[:50]}" → +{len(variants)} variants (class total so far: {intent_count})')

    print(f'  ✅ {intent.upper()} done: {intent_count} examples')

df_raw = pd.DataFrame(records)
df_raw = df_raw.drop_duplicates(subset='utterance').reset_index(drop=True)

print(f'\n📊 Dataset shape: {df_raw.shape}')
print(df_raw['intent'].value_counts())

# Quick length distribution check
df_raw['word_count'] = df_raw['utterance'].str.split().str.len()
print(f'\nLength stats (words):')
print(df_raw.groupby('intent')['word_count'].describe()[['min','mean','max']].round(1))


🔄 [DEBUG] — 12 seeds × 14 variants
  [1/12] "fix the null pointer exception in login service" → +14 variants (class total so far: 15)
  [2/12] "why is my authentication module throwing a 403 err" → +14 variants (class total so far: 30)
  [3/12] "debug the memory leak in the payment processor" → +14 variants (class total so far: 45)
  [4/12] "the api is returning 500 on every post request" → +14 variants (class total so far: 60)
  [5/12] "find the bug causing infinite loop in data pipelin" → +15 variants (class total so far: 76)
  [6/12] "trace the runtime exception in the order handler" → +14 variants (class total so far: 91)
  [7/12] "resolve the type error in user profile component" → +14 variants (class total so far: 106)
  [8/12] "the database connection keeps timing out fix it" → +14 variants (class total so far: 121)
  [9/12] "the mobile app crashes on android when uploading a" → +14 variants (class total so far: 136)
  [10/12] "my docker container exits with code 137 every few 

In [8]:
# ── Load saved dataset from CSV (skip re-generation)   
df_raw = pd.read_csv(r'd:\NLP\devflow_dataset.csv')

print(f'✅ Loaded dataset: {df_raw.shape}')
print(df_raw['intent'].value_counts())

df_raw['word_count'] = df_raw['utterance'].str.split().str.len()
print(f'\nLength stats (words):')
print(df_raw.groupby('intent')['word_count'].describe()[['min','mean','max']].round(1))

✅ Loaded dataset: (1264, 18)
intent
refactor    182
debug       181
test        181
generate    180
explain     180
scaffold    180
document    180
Name: count, dtype: int64

Length stats (words):
          min  mean   max
intent                   
debug     2.0  13.1  34.0
document  2.0  13.0  40.0
explain   2.0  12.1  31.0
generate  3.0  13.0  36.0
refactor  2.0  13.2  36.0
scaffold  2.0  16.2  47.0
test      3.0  13.2  28.0


CSV already has language, framework, error_type columns from when it was first saved.
<br/> :. no need to run the entity annotation cell again n again.

In [9]:
# ── Lightweight entity annotation via regex heuristics ───────────────────────
# These are noisy silver labels — used for entity extraction training/eval

# Run the following  when touching the dataset for the first time
# LANG_PATTERNS      = r'\b(python|typescript|javascript|java|go|rust|kotlin|swift|cpp|c\+\+|ruby|php|dart|bash)\b'
# FRAMEWORK_PATTERNS = r'\b(react|nextjs|angular|vue|express|django|fastapi|spring|flutter|rails|laravel|nestjs|terraform|grafana|prometheus)\b'
# ERROR_PATTERNS     = r'\b(null\s*pointer|nullpointerexception|type\s*error|runtime\s*exception|key\s*error|404|403|500|137|index\s*out\s*of\s*bounds|stack\s*overflow|segfault|memory\s*leak|infinite\s*loop|timeout)\b'

# def extract_entities_regex(text: str) -> dict:
#     t = text.lower()
#     lang = re.search(LANG_PATTERNS, t)
#     fw   = re.search(FRAMEWORK_PATTERNS, t)
#     err  = re.search(ERROR_PATTERNS, t)
#     return {
#         'language':   lang.group(0) if lang else None,
#         'framework':  fw.group(0)   if fw   else None,
#         'error_type': err.group(0)  if err  else None,
#     }

# entity_cols = df_raw['utterance'].apply(extract_entities_regex).apply(pd.Series)
# df = pd.concat([df_raw, entity_cols], axis=1)

# # ── Export ────────────────────────────────────────────────────────────────────
# EXPORT_PATH = r'd:\NLP\devflow_dataset.csv'
# df.to_csv(EXPORT_PATH, index=False)

df = df_raw.copy()

# print(f'✅ Dataset saved: {EXPORT_PATH}')
print(f'✅ Dataset ready: {df.shape}')
print(f'   Rows    : {len(df)}')
print(f'   Columns : {list(df.columns)}')
print(f'\nClass distribution:')
print(df["intent"].value_counts().to_string())
print(f'\nEntity coverage:')
print(f'   language   filled: {df["language"].notna().sum()} / {len(df)} ({df["language"].notna().mean()*100:.1f}%)')
print(f'   framework  filled: {df["framework"].notna().sum()} / {len(df)} ({df["framework"].notna().mean()*100:.1f}%)')
print(f'   error_type filled: {df["error_type"].notna().sum()} / {len(df)} ({df["error_type"].notna().mean()*100:.1f}%)')
print(f'\nSample row:')
print(df.sample(1).to_dict(orient='records')[0])

✅ Dataset ready: (1264, 18)
   Rows    : 1264
   Columns : ['utterance', 'intent', 'word_count', 'language', 'framework', 'error_type', 'language.1', 'framework.1', 'error_type.1', 'language.2', 'framework.2', 'error_type.2', 'language.3', 'framework.3', 'error_type.3', 'language.4', 'framework.4', 'error_type.4']

Class distribution:
intent
refactor    182
debug       181
test        181
generate    180
explain     180
scaffold    180
document    180

Entity coverage:
   language   filled: 164 / 1264 (13.0%)
   framework  filled: 306 / 1264 (24.2%)
   error_type filled: 123 / 1264 (9.7%)

Sample row:
{'utterance': 'can you create a runbook for when our payment service goes down', 'intent': 'document', 'word_count': 12, 'language': nan, 'framework': nan, 'error_type': nan, 'language.1': nan, 'framework.1': nan, 'error_type.1': nan, 'language.2': nan, 'framework.2': nan, 'error_type.2': nan, 'language.3': nan, 'framework.3': nan, 'error_type.3': nan, 'language.4': nan, 'framework.4': na

## 3. Preprocessing Pipeline (6 Steps)

In [10]:
# ── Full preprocessing pipeline ───────────────────────────────────────────────

lemmatizer  = WordNetLemmatizer()
nltk_stops  = set(stopwords.words('english'))

# Step — Custom stop-word list (domain modifications)
# REMOVE from stop-list: words that carry intent/entity signal
RETAIN_WORDS = {'not', 'no', 'why', 'how', 'fix', 'run', 'what', 'where', 'when'}
CUSTOM_STOPS = nltk_stops - RETAIN_WORDS

# Filler words (voice-specific noise)
FILLERS = re.compile(
    r'\b(um|uh|like|you\s+know|basically|literally|hey|okay|so|just|kind\s+of|sort\s+of)\b'
)

# Contractions map
CONTRACTIONS = {
    "it's": 'it is', "don't": 'do not', "doesn't": 'does not',
    "can't": 'cannot', "won't": 'will not', "isn't": 'is not',
    "aren't": 'are not', "wasn't": 'was not', "weren't": 'were not',
    "i'm": 'i am', "i've": 'i have', "i'll": 'i will',
    "we're": 'we are', "they're": 'they are', "there's": 'there is',
    "that's": 'that is', "what's": 'what is', "let's": 'let us',
    "wouldn't": 'would not', "shouldn't": 'should not',
    "couldn't": 'could not', "didn't": 'did not',
}

def expand_contractions(text: str) -> str:
    for contraction, expansion in CONTRACTIONS.items():
        text = text.replace(contraction, expansion)
    return text

# POS tag → WordNet POS (for lemmatiser)
def get_wordnet_pos(treebank_tag: str) -> str:
    if treebank_tag.startswith('J'): return wordnet.ADJ
    if treebank_tag.startswith('V'): return wordnet.VERB
    if treebank_tag.startswith('N'): return wordnet.NOUN
    if treebank_tag.startswith('R'): return wordnet.ADV
    return wordnet.NOUN  # default


def preprocess(text: str, return_steps: bool = False) -> str | dict:
    """
    Full 6-step preprocessing pipeline.
    If return_steps=True, returns dict with intermediate outputs (for ablation).
    """
    steps = {}

    # ── STEP 1: Normalisation (voice-specific) ────────────────────────────────
    s = text.lower()                        # lowercase
    s = expand_contractions(s)              # contractions
    s = FILLERS.sub('', s)                  # remove fillers
    s = re.sub(r'[^\w\s]', ' ', s)         # strip punctuation
    s = re.sub(r'\s+', ' ', s).strip()     # collapse whitespace
    steps['normalised'] = s

    # ── STEP 2: Tokenisation ──────────────────────────────────────────────────
    tokens = word_tokenize(s)
    steps['tokens'] = tokens

    # ── STEP 3: Stop-word removal (custom list) ───────────────────────────────
    tokens = [t for t in tokens if t not in CUSTOM_STOPS]
    steps['after_stopword_removal'] = tokens

    # ── STEP 4: POS tagging (before lemmatisation — needed for POS-aware lemma) 
    tagged = pos_tag(tokens)
    steps['pos_tagged'] = tagged

    # ── STEP 5: Lemmatisation (POS-aware, chosen over stemming) ──────────────
    lemmatised = [
        lemmatizer.lemmatize(word, get_wordnet_pos(tag))
        for word, tag in tagged
    ]
    steps['lemmatised'] = lemmatised

    # ── STEP 6: Rejoin for vectoriser ─────────────────────────────────────────
    result = ' '.join(lemmatised)
    steps['final'] = result

    return steps if return_steps else result


# ── Demo 
example = 'um fix the null pointer in login service'
steps   = preprocess(example, return_steps=True)
print('Pipeline demo:')
for k, v in steps.items():
    print(f'  {k:30s}: {v}')

Pipeline demo:
  normalised                    : fix the null pointer in login service
  tokens                        : ['fix', 'the', 'null', 'pointer', 'in', 'login', 'service']
  after_stopword_removal        : ['fix', 'null', 'pointer', 'login', 'service']
  pos_tagged                    : [('fix', 'JJ'), ('null', 'JJ'), ('pointer', 'NN'), ('login', 'NN'), ('service', 'NN')]
  lemmatised                    : ['fix', 'null', 'pointer', 'login', 'service']
  final                         : fix null pointer login service


In [11]:
# ── Apply preprocessing to full dataset 
df['processed'] = df['utterance'].apply(preprocess)

# Encode intent labels
intent2id = {intent: i for i, intent in enumerate(INTENTS)}
id2intent  = {i: intent for intent, i in intent2id.items()}
df['label'] = df['intent'].map(intent2id)

# Train / Val / Test split  70 / 15 / 15
df_train, df_temp  = train_test_split(df, test_size=0.30, stratify=df['label'], random_state=SEED)
df_val,   df_test  = train_test_split(df_temp, test_size=0.50, stratify=df_temp['label'], random_state=SEED)

print(f'Train: {len(df_train)} | Val: {len(df_val)} | Test: {len(df_test)}')
print('\nClass distribution (train):')
print(df_train['intent'].value_counts())

Train: 884 | Val: 190 | Test: 190

Class distribution (train):
intent
refactor    127
test        127
document    126
explain     126
scaffold    126
debug       126
generate    126
Name: count, dtype: int64


## 4. Model A — LinearSVC Baseline

In [12]:
# ── LinearSVC with TF-IDF (calibrated for confidence scores) ─────────────────

svc_pipeline = Pipeline([
    ('tfidf', TfidfVectorizer(
        ngram_range=(1, 2),     # bigrams critical for developer terms
        max_features=5000,
        sublinear_tf=True,
    )),
    # CalibratedClassifierCV wraps LinearSVC to give predict_proba
    ('clf', CalibratedClassifierCV(
        LinearSVC(max_iter=2000, random_state=SEED), cv=3
    )),
])

svc_pipeline.fit(df_train['processed'], df_train['label'])

# Evaluate
svc_preds = svc_pipeline.predict(df_test['processed'])
svc_f1    = f1_score(df_test['label'], svc_preds, average='macro')

print('=== Model A: LinearSVC ===')
print(f'Macro F1 (test): {svc_f1:.4f}\n')
print(classification_report(df_test['label'], svc_preds, target_names=INTENTS))

=== Model A: LinearSVC ===
Macro F1 (test): 0.9948

              precision    recall  f1-score   support

       debug       0.97      1.00      0.98        28
    generate       1.00      1.00      1.00        27
    refactor       1.00      0.96      0.98        27
     explain       1.00      1.00      1.00        27
    scaffold       1.00      1.00      1.00        27
        test       1.00      1.00      1.00        27
    document       1.00      1.00      1.00        27

    accuracy                           0.99       190
   macro avg       1.00      0.99      0.99       190
weighted avg       0.99      0.99      0.99       190



In [13]:
# ── Save LinearSVC pipeline ───────────────────────────────────────────────────
import pickle
from pathlib import Path

Path('devflow_artefacts').mkdir(exist_ok=True)

with open('devflow_artefacts/svc_pipeline.pkl', 'wb') as f:
    pickle.dump(svc_pipeline, f)

print(f'✅ LinearSVC saved → devflow_artefacts/svc_pipeline.pkl')
print(f'   Macro F1 (test): {svc_f1:.4f}')

✅ LinearSVC saved → devflow_artefacts/svc_pipeline.pkl
   Macro F1 (test): 0.9948


## 5. Model B — DistilBERT Fine-tuning

Why DistilBERT?

The trained DistilBERT model is roughly 250-260 MB in size (for distilbert-base-uncased), containing 66 million parameters 
- It is 40% smaller than BERT-base, featuring 6 layers (instead of 12) with a 768-hidden size
- It operates 60% faster while retaining roughly 97% of BERT's performance 
- Key Model Statistics:
    Total Parameters: ~66 Million.
    Storage Size: ~250–260 MB (FP32).
    Architecture: 6 layers, 768 hidden size, 12 attention heads 
- VRAM Required: ~1-2 GB for inference 

DistilBERT achieves this reduction by using knowledge distillation to transfer knowledge from a teacher BERT model to a smaller student, optimized for faster inference 

### Loading the existing DistilBERT

In [14]:
# ── Load DistilBERT tokeniser from checkpoint ─────────────────────────────────
DISTILBERT_PATH = r'd:\NLP\devflow_distilbert\checkpoint-112'   # epoch 2, val F1=0.9682

tokenizer = AutoTokenizer.from_pretrained('distilbert-base-uncased')  # tokeniser from HuggingFace

def tokenize_fn(batch):
    return tokenizer(
        batch['utterance'],
        truncation=True,
        padding='max_length',
        max_length=64,
    )

def df_to_hf(df_split):
    return Dataset.from_dict({
        'utterance': df_split['utterance'].tolist(),
        'labels':    df_split['label'].tolist(),
    })

hf_train = df_to_hf(df_train).map(tokenize_fn, batched=True)
hf_val   = df_to_hf(df_val).map(tokenize_fn, batched=True)
hf_test  = df_to_hf(df_test).map(tokenize_fn, batched=True)

print(f'✅ Tokeniser ready')
print(f'   Checkpoint      : {DISTILBERT_PATH}')
print(f'   Val Macro F1    : 0.9682 (epoch 2)')
print(f'   Train: {len(hf_train)} | Val: {len(hf_val)} | Test: {len(hf_test)}')

Map:   0%|          | 0/884 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

✅ Tokeniser ready
   Checkpoint      : d:\NLP\devflow_distilbert\checkpoint-112
   Val Macro F1    : 0.9682 (epoch 2)
   Train: 884 | Val: 190 | Test: 190


In [ ]:
import sys

print("Test 1: import safetensors"); sys.stdout.flush()
from safetensors.torch import load_file

print("Test 2: import DistilBert classes"); sys.stdout.flush()
from transformers import DistilBertConfig, DistilBertForSequenceClassification

print("Test 3: read config.json"); sys.stdout.flush()
config = DistilBertConfig.from_json_file(f'{DISTILBERT_PATH}/config.json')

print("Test 4: build architecture"); sys.stdout.flush()
model = DistilBertForSequenceClassification(config)

print("Test 5: load weights file"); sys.stdout.flush()
weights = load_file(f'{DISTILBERT_PATH}/model.safetensors')

print("Test 6: apply weights"); sys.stdout.flush()
model.load_state_dict(weights)

print("✅ All done!")

In [ ]:
# ── Load DistilBERT — fully offline, no from_pretrained ───────────────────────
from transformers import DistilBertForSequenceClassification, DistilBertConfig
from safetensors.torch import load_file
import sys

print("Step 1/3: Reading config.json...")
sys.stdout.flush()

# Load config directly from JSON — zero network calls
config = DistilBertConfig.from_json_file(f'{DISTILBERT_PATH}/config.json')

print("Step 2/3: Building model architecture...")
sys.stdout.flush()

# Build the model from config — zero network calls  
model = DistilBertForSequenceClassification(config)

print("Step 3/3: Loading weights from model.safetensors (267MB)...")
sys.stdout.flush()

# Load fine-tuned weights directly from file — zero network calls
weights = load_file(f'{DISTILBERT_PATH}/model.safetensors')
model.load_state_dict(weights)
model.to(DEVICE)
model.eval()

total_params = sum(p.numel() for p in model.parameters())
print(f'\n✅ DistilBERT loaded successfully')
print(f'   Checkpoint : checkpoint-112 (epoch 2, val F1=0.9682)')
print(f'   Device     : {DEVICE}')
print(f'   Parameters : {total_params:,}')

In [ ]:
# ── Load DistilBERT weights from checkpoint (offline, no HuggingFace ping) ───
import sys

print("Loading model weights from disk...")
sys.stdout.flush()

model = AutoModelForSequenceClassification.from_pretrained(
    DISTILBERT_PATH,
    num_labels=len(INTENTS),
    id2label=id2intent,
    label2id=intent2id,
    ignore_mismatched_sizes=True,
    local_files_only=True,      # ← prevents HuggingFace network call
)
model.to(DEVICE)
model.eval()

print(f'✅ DistilBERT loaded from checkpoint-112 (epoch 2, val F1=0.9682)')
print(f'   Device  : {DEVICE}')
print(f'   Params  : {sum(p.numel() for p in model.parameters()):,}')

In [ ]:
# ── DistilBERT evaluation — batch inference on test set ───────────────────────
import torch.nn.functional as F
from torch.utils.data import DataLoader

def bert_predict_batch(df_split, batch_size=32):
    all_preds, all_labels = [], []
    hf = df_to_hf(df_split).map(tokenize_fn, batched=True)
    hf.set_format(type='torch', columns=['input_ids', 'attention_mask', 'labels'])
    loader = DataLoader(hf, batch_size=batch_size)

    model.eval()
    with torch.no_grad():
        for batch in loader:
            input_ids      = batch['input_ids'].to(DEVICE)
            attention_mask = batch['attention_mask'].to(DEVICE)
            labels         = batch['labels']
            logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
            preds  = logits.argmax(dim=-1).cpu().numpy()
            all_preds.extend(preds)
            all_labels.extend(labels.numpy())

    return np.array(all_preds), np.array(all_labels)


bert_preds, bert_labels = bert_predict_batch(df_test)
bert_f1 = f1_score(bert_labels, bert_preds, average='macro')

print('=== Model B: DistilBERT (checkpoint-112, epoch 2) ===')
print(f'Macro F1 (test): {bert_f1:.4f}\n')
print(classification_report(bert_labels, bert_preds, target_names=INTENTS))

### NOTE: DistilBERT already Trained using Colab for compute power so no need to re-run the training

In [16]:
# ── DistilBERT fine-tuning ────────────────────────────────────────────────────
# Using RAW utterances (not preprocessed) — BERT handles its own tokenisation

MODEL_NAME = 'distilbert-base-uncased'
tokenizer  = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_fn(batch):
    return tokenizer(
        batch['utterance'],
        truncation=True,
        padding='max_length',
        max_length=64,     # developer utterances are short
    )

# Build HuggingFace Datasets
def df_to_hf(df_split):
    return Dataset.from_dict({
        'utterance': df_split['utterance'].tolist(),
        'labels':    df_split['label'].tolist(),
    })

hf_train = df_to_hf(df_train).map(tokenize_fn, batched=True)
hf_val   = df_to_hf(df_val).map(tokenize_fn, batched=True)
hf_test  = df_to_hf(df_test).map(tokenize_fn, batched=True)

print(f'✅ Datasets tokenised | Train: {len(hf_train)} | Val: {len(hf_val)} | Test: {len(hf_test)}')

config.json:   0%|          | 0.00/483 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Map:   0%|          | 0/884 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

Map:   0%|          | 0/190 [00:00<?, ? examples/s]

✅ Datasets tokenised | Train: 884 | Val: 190 | Test: 190


In [ ]:
# ── Training ──────────────────────────────────────────────────────────────────

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=len(INTENTS),
    id2label=id2intent,
    label2id=intent2id,
)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {'macro_f1': f1_score(labels, preds, average='macro')}

training_args = TrainingArguments(
    output_dir          = './devflow_distilbert',
    num_train_epochs    = 8,
    per_device_train_batch_size = 16,
    per_device_eval_batch_size  = 32,
    learning_rate       = 3e-5,
    weight_decay        = 0.01,
    warmup_ratio        = 0.1,
    eval_strategy       = 'epoch',
    save_strategy       = 'epoch',
    load_best_model_at_end = True,
    metric_for_best_model  = 'macro_f1',
    logging_steps       = 20,
    fp16                = (DEVICE == 'cuda'),
    seed                = SEED,
    report_to           = 'none',   # swap to 'wandb' if you set up W&B
)

trainer = Trainer(
    model           = model,
    args            = training_args,
    train_dataset   = hf_train,
    eval_dataset    = hf_val,
    compute_metrics = compute_metrics,
)

print('Starting DistilBERT fine-tuning...')
trainer.train()
print('Training complete')

In [ ]:
# ── DistilBERT evaluation on test set ────────────────────────────────────────

bert_preds_raw = trainer.predict(hf_test)
bert_preds     = np.argmax(bert_preds_raw.predictions, axis=-1)
bert_labels    = bert_preds_raw.label_ids
bert_f1        = f1_score(bert_labels, bert_preds, average='macro')

print('=== Model B: DistilBERT ===')
print(f'Macro F1 (test): {bert_f1:.4f}\n')
print(classification_report(bert_labels, bert_preds, target_names=INTENTS))

## 6. Entity Extractor — spaCy EntityRuler

In [ ]:
# ── spaCy EntityRuler with custom developer patterns ─────────────────────────

nlp = spacy.load('en_core_web_sm')

ruler = nlp.add_pipe('entity_ruler', before='ner')

# Pattern definitions
ENTITY_PATTERNS = [
    # Languages
    *[{'label': 'LANG',      'pattern': lang} for lang in [
        'python', 'typescript', 'javascript', 'java', 'go', 'rust',
        'kotlin', 'swift', 'ruby', 'php', 'c++', 'cpp', 'scala', 'elixir'
    ]],
    # Frameworks
    *[{'label': 'FRAMEWORK', 'pattern': fw} for fw in [
        'react', 'nextjs', 'next js', 'angular', 'vue', 'express',
        'django', 'fastapi', 'spring', 'flutter', 'rails', 'laravel',
        'nestjs', 'nest js', 'nuxt', 'svelte', 'remix', 'astro', 'gin',
        'fiber', 'axum', 'actix'
    ]],
    # Error types
    {'label': 'ERROR', 'pattern': [{'LOWER': 'null'}, {'LOWER': 'pointer'}]},
    {'label': 'ERROR', 'pattern': [{'LOWER': 'null'}, {'LOWER': 'pointer'}, {'LOWER': 'exception'}]},
    {'label': 'ERROR', 'pattern': [{'LOWER': 'type'}, {'LOWER': 'error'}]},
    {'label': 'ERROR', 'pattern': [{'LOWER': 'runtime'}, {'LOWER': 'exception'}]},
    {'label': 'ERROR', 'pattern': [{'LOWER': 'index'}, {'LOWER': 'out'}, {'LOWER': 'of'}, {'LOWER': 'bounds'}]},
    {'label': 'ERROR', 'pattern': [{'LOWER': 'memory'}, {'LOWER': 'leak'}]},
    {'label': 'ERROR', 'pattern': [{'LOWER': 'stack'}, {'LOWER': 'overflow'}]},
    {'label': 'ERROR', 'pattern': [{'LOWER': 'infinite'}, {'LOWER': 'loop'}]},
    {'label': 'ERROR', 'pattern': [{'TEXT': {'REGEX': r'^(404|403|500|502|503)$'}}]},
    {'label': 'ERROR', 'pattern': 'segfault'},
    {'label': 'ERROR', 'pattern': 'timeout'},
    # Testing frameworks (useful for TEST intent)
    *[{'label': 'FRAMEWORK', 'pattern': t} for t in [
        'jest', 'pytest', 'cypress', 'junit', 'mocha', 'chai', 'vitest'
    ]],
]

ruler.add_patterns(ENTITY_PATTERNS)

def extract_entities_spacy(text: str) -> dict:
    doc = nlp(text.lower())
    entities = {'language': None, 'framework': None, 'error_type': None, 'component': None}

    for ent in doc.ents:
        if ent.label_ == 'LANG'      and not entities['language']:   entities['language']   = ent.text
        if ent.label_ == 'FRAMEWORK' and not entities['framework']:  entities['framework']  = ent.text
        if ent.label_ == 'ERROR'     and not entities['error_type']: entities['error_type'] = ent.text

    # Heuristic: last NOUN chunk not matched by above → component
    noun_chunks = [chunk.text for chunk in doc.noun_chunks
                   if chunk.text not in (entities.values())]
    if noun_chunks:
        entities['component'] = noun_chunks[-1]

    return entities


# Demo
test_utt = 'uh fix the null pointer in the login service using spring'
print('Entity extraction demo:')
print(f'  Input   : {test_utt}')
print(f'  Entities: {extract_entities_spacy(test_utt)}')

## 7. Full Inference Pipeline → JSON Output

In [ ]:
# ── Full inference: utterance → JSON ─────────────────────────────────────────
# Uses DistilBERT for intent + confidence; spaCy for entities
# Falls back to LinearSVC if BERT unavailable

import torch.nn.functional as F

def predict_intent_bert(utterance: str) -> tuple[str, float]:
    """Returns (intent_label, confidence_score) using DistilBERT."""
    enc = tokenizer(
        utterance, return_tensors='pt',
        truncation=True, padding='max_length', max_length=64
    ).to(DEVICE)
    model.eval()
    with torch.no_grad():
        logits = model(**enc).logits
    probs     = F.softmax(logits, dim=-1).squeeze()
    pred_id   = probs.argmax().item()
    confidence = probs[pred_id].item()
    return id2intent[pred_id], round(confidence, 4)


def predict_intent_svc(utterance: str) -> tuple[str, float]:
    """Fallback: LinearSVC with calibrated probabilities."""
    processed = preprocess(utterance)
    proba     = svc_pipeline.predict_proba([processed])[0]
    pred_id   = int(np.argmax(proba))
    return id2intent[pred_id], round(float(proba[pred_id]), 4)


def normalise_for_output(text: str) -> str:
    """Clean display version (Step 1 only, no stop-word removal)."""
    s = text.lower()
    s = expand_contractions(s)
    s = FILLERS.sub('', s)
    s = re.sub(r'[^\w\s]', ' ', s)
    return re.sub(r'\s+', ' ', s).strip()


def run_pipeline(utterance: str, use_bert: bool = True) -> dict:
    """End-to-end inference. Returns the target JSON schema."""
    if use_bert:
        intent, confidence = predict_intent_bert(utterance)
    else:
        intent, confidence = predict_intent_svc(utterance)

    entities = extract_entities_spacy(utterance)

    return {
        'raw_utterance':    utterance,
        'normalized':       normalise_for_output(utterance),
        'intent':           intent,
        'confidence_score': confidence,
        'entities':         entities,
    }


# ── Live demos ────────────────────────────────────────────────────────────────
demo_utterances = [
    'um fix the null pointer in login service',
    'uh write a react component for the dashboard like',
    'basically refactor the auth module to use dependency injection',
    'explain how the caching layer works',
    'set up a new django project with postgres you know',
    'write unit tests for the payment service using jest',
    'generate jsdoc for all the api endpoints',
]

for utt in demo_utterances:
    result = run_pipeline(utt)
    print(json.dumps(result, indent=2))
    print()

## 8. Evaluation — Macro F1, Ablation Study, Cohen's Kappa

In [ ]:
# ── Side-by-side model comparison ────────────────────────────────────────────

print('=' * 50)
print('         MODEL COMPARISON SUMMARY')
print('=' * 50)
print(f'  LinearSVC  macro F1 : {svc_f1:.4f}')
print(f'  DistilBERT macro F1 : {bert_f1:.4f}')
print(f'  Delta               : {bert_f1 - svc_f1:+.4f}')

# Per-class breakdown comparison
svc_per  = precision_recall_fscore_support(df_test['label'], svc_preds,  average=None, labels=list(range(7)))
bert_per = precision_recall_fscore_support(bert_labels,      bert_preds, average=None, labels=list(range(7)))

comparison_df = pd.DataFrame({
    'intent':           INTENTS,
    'svc_f1':           svc_per[2],
    'bert_f1':          bert_per[2],
    'delta':            bert_per[2] - svc_per[2],
})
print('\nPer-class F1:')
print(comparison_df.to_string(index=False))

In [ ]:
# ── Ablation Study ────────────────────────────────────────────────────────────
# Remove one preprocessing step at a time, measure macro F1 delta on LinearSVC
# (SVC used for ablation: deterministic, fast, no GPU needed)

def preprocess_ablated(text: str, skip_step: str | None = None) -> str:
    """Same pipeline but skip one named step."""

    # Step 1: Normalisation
    if skip_step == 'normalisation':
        s = text
    else:
        s = text.lower()
        s = expand_contractions(s)
        s = FILLERS.sub('', s)
        s = re.sub(r'[^\w\s]', ' ', s)
        s = re.sub(r'\s+', ' ', s).strip()

    # Step 2: Tokenisation (always)
    tokens = word_tokenize(s)

    # Step 3: Stop-word removal
    if skip_step != 'stopword_removal':
        tokens = [t for t in tokens if t not in CUSTOM_STOPS]

    # Step 4: POS tagging
    tagged = pos_tag(tokens)

    # Step 5: Lemmatisation
    if skip_step == 'lemmatisation':
        lemmatised = [word for word, _ in tagged]
    else:
        lemmatised = [lemmatizer.lemmatize(w, get_wordnet_pos(t)) for w, t in tagged]

    return ' '.join(lemmatised)


def ablation_svc(skip_step: str | None) -> float:
    """Train + evaluate LinearSVC with one step removed."""
    X_train = df_train['utterance'].apply(lambda x: preprocess_ablated(x, skip_step))
    X_test  = df_test['utterance'].apply(lambda x: preprocess_ablated(x, skip_step))

    # For TF-IDF ablation, keep everything else the same
    if skip_step == 'bigrams':
        vect = TfidfVectorizer(ngram_range=(1,1), max_features=5000, sublinear_tf=True)
    else:
        vect = TfidfVectorizer(ngram_range=(1,2), max_features=5000, sublinear_tf=True)

    pipe = Pipeline([
        ('tfidf', vect),
        ('clf',   CalibratedClassifierCV(LinearSVC(max_iter=2000, random_state=SEED), cv=3))
    ])
    pipe.fit(X_train, df_train['label'])
    preds = pipe.predict(X_test)
    return f1_score(df_test['label'], preds, average='macro')


# Run ablation
ablation_steps = {
    'Full pipeline':     None,
    '− Normalisation':   'normalisation',
    '− Stop-word removal': 'stopword_removal',
    '− Lemmatisation':   'lemmatisation',
    '− Bigrams':         'bigrams',
}

print('Running ablation study...')
ablation_results = {}
for name, skip in ablation_steps.items():
    f1 = ablation_svc(skip)
    ablation_results[name] = f1
    print(f'  {name:30s}: {f1:.4f}')

baseline = ablation_results['Full pipeline']
print('\nF1 delta vs full pipeline:')
for name, f1 in ablation_results.items():
    delta = f1 - baseline
    print(f'  {name:30s}: {delta:+.4f}')

In [ ]:
# ── Paired t-test across 10 randomised test splits ────────────────────────────
# Tests whether DistilBERT consistently outperforms LinearSVC

N_SPLITS = 10
svc_scores, bert_scores = [], []

print('Running paired t-test across 10 randomised splits...')
for seed_i in range(N_SPLITS):
    _, df_test_i = train_test_split(df, test_size=0.15, stratify=df['label'], random_state=seed_i)

    # SVC
    s_preds  = svc_pipeline.predict(df_test_i['processed'])
    svc_scores.append(f1_score(df_test_i['label'], s_preds, average='macro'))

    # BERT
    hf_i  = df_to_hf(df_test_i).map(tokenize_fn, batched=True)
    b_raw = trainer.predict(hf_i)
    b_pred = np.argmax(b_raw.predictions, axis=-1)
    bert_scores.append(f1_score(b_raw.label_ids, b_pred, average='macro'))

t_stat, p_value = stats.ttest_rel(bert_scores, svc_scores)
print(f'\n  SVC  mean F1 : {np.mean(svc_scores):.4f} ± {np.std(svc_scores):.4f}')
print(f'  BERT mean F1 : {np.mean(bert_scores):.4f} ± {np.std(bert_scores):.4f}')
print(f'  t-statistic  : {t_stat:.4f}')
print(f'  p-value      : {p_value:.4f}')
print(f'  Significant  : {"YES" if p_value < 0.05 else "NO"} (α = 0.05)')

In [ ]:
# ── Cohen's Kappa — automated vs human eval on 10% sample ────────────────────
# In practice: a human annotator labels 10% of test set → compare to model
# Here we simulate human labels with controlled noise for demonstration

sample_size  = max(10, int(0.10 * len(df_test)))
df_sample    = df_test.sample(n=sample_size, random_state=SEED)

model_labels = svc_pipeline.predict(df_sample['processed']).tolist()

# Simulate human labels: 92% agreement (realistic inter-rater scenario)
np.random.seed(SEED)
human_labels = [
    lbl if random.random() > 0.08 else random.choice(list(range(7)))
    for lbl in model_labels
]

kappa = cohen_kappa_score(model_labels, human_labels)
print(f'Cohen\'s Kappa (automated vs human, 10% sample): {kappa:.4f}')
print(f'Interpretation: {"Substantial" if kappa > 0.6 else "Moderate" if kappa > 0.4 else "Fair"} agreement')
print('\nNote: Replace simulated human_labels with your actual human annotations.')

## 9. Save Artefacts for Web App Integration

In [ ]:
# ── Save everything your web app needs ───────────────────────────────────────
import pickle

Path('devflow_artefacts').mkdir(exist_ok=True)

# 1. LinearSVC pipeline (sklearn pickle)
with open('devflow_artefacts/svc_pipeline.pkl', 'wb') as f:
    pickle.dump(svc_pipeline, f)
print('✅ Saved: svc_pipeline.pkl')

# 2. DistilBERT model + tokeniser
model.save_pretrained('devflow_artefacts/distilbert')
tokenizer.save_pretrained('devflow_artefacts/distilbert')
print('✅ Saved: distilbert/')

# 3. spaCy model with EntityRuler
nlp.to_disk('devflow_artefacts/spacy_ner')
print('✅ Saved: spacy_ner/')

# 4. Label maps
with open('devflow_artefacts/label_maps.json', 'w') as f:
    json.dump({'intent2id': intent2id, 'id2intent': id2intent}, f, indent=2)
print('✅ Saved: label_maps.json')

# 5. Dataset
df.to_csv('devflow_artefacts/devflow_dataset.csv', index=False)
print('✅ Saved: devflow_dataset.csv')

# 6. List saved artefacts
import os
print('\n📦 All artefacts saved to devflow_artefacts/:')
for root, dirs, files_list in os.walk('devflow_artefacts'):
    for fname in files_list:
        fpath = os.path.join(root, fname)
        size_kb = os.path.getsize(fpath) / 1024
        print(f'   {fpath} ({size_kb:.1f} KB)')

In [ ]:
# ── Web app integration snippet ───────────────────────────────────────────────
# Copy this into your web app's model loader

INTEGRATION_SNIPPET = '''
# ── devflow_model.py (drop into your web app) ─────────────────────────────────
import json, pickle, re, spacy
import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer
from nltk.corpus import stopwords
from nltk import pos_tag, wordnet

# ── Load artefacts ────────────────────────────────────────────────────────────
with open('devflow_artefacts/label_maps.json') as f:
    maps = json.load(f)
    id2intent = {int(k): v for k, v in maps['id2intent'].items()}

tokenizer = AutoTokenizer.from_pretrained('devflow_artefacts/distilbert')
model     = AutoModelForSequenceClassification.from_pretrained('devflow_artefacts/distilbert')
nlp       = spacy.load('devflow_artefacts/spacy_ner')
DEVICE    = 'cuda' if torch.cuda.is_available() else 'cpu'
model.to(DEVICE).eval()

def run_pipeline(utterance: str) -> dict:
    enc = tokenizer(utterance, return_tensors='pt',
                    truncation=True, padding='max_length', max_length=64).to(DEVICE)
    with torch.no_grad():
        logits = model(**enc).logits
    probs      = F.softmax(logits, dim=-1).squeeze()
    pred_id    = probs.argmax().item()
    confidence = round(probs[pred_id].item(), 4)

    doc      = nlp(utterance.lower())
    entities = {"language": None, "framework": None, "error_type": None, "component": None}
    for ent in doc.ents:
        if ent.label_ == "LANG"      and not entities["language"]:   entities["language"]   = ent.text
        if ent.label_ == "FRAMEWORK" and not entities["framework"]:  entities["framework"]  = ent.text
        if ent.label_ == "ERROR"     and not entities["error_type"]: entities["error_type"] = ent.text

    return {
        "raw_utterance":    utterance,
        "normalized":       utterance.lower(),
        "intent":           id2intent[pred_id],
        "confidence_score": confidence,
        "entities":         entities,
    }
'''

print(INTEGRATION_SNIPPET)

with open('devflow_artefacts/devflow_model.py', 'w') as f:
    f.write(INTEGRATION_SNIPPET.strip())
print('✅ Integration snippet saved to devflow_artefacts/devflow_model.py')